Creación de DB mediante vectorización de PDF con Ollama

In [1]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
import re, os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

DB_DIR = "./chroma_db_rag"
PDF_PATH = '../Mobile_Site_Speed_Playbook.pdf'

embeddings = OllamaEmbeddings(model="paraphrase-multilingual:latest")

if os.path.exists(DB_DIR):
    vectorstore = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)
    print("--- Base de Datos existente cargada---")
else:
    print("--- Procesando PDF ---")
    loader = PyPDFLoader(PDF_PATH)
    data = loader.load()

    def clean_text(text):
        text = re.sub(r'\n\d+\s*\n', '\n', text)
        text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
        return " ".join(text.split())

    data_filtered = [doc for doc in data if len(doc.page_content.strip()) > 10]
    for doc in data_filtered:
        doc.page_content = clean_text(doc.page_content)

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,        
        chunk_overlap=100,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    chunks = text_splitter.split_documents(data_filtered)

    vectorstore = Chroma.from_documents(
        documents=chunks, embedding=embeddings, persist_directory=DB_DIR
    )
    print(f"--- DB creada con {len(chunks)} chunks ---")


retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 8, "fetch_k": 10, "lambda_mult": 0.7})

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


--- Base de Datos existente cargada---


In [2]:
from langchain_core.tools import tool
from IPython.display import display, Markdown
from langchain.agents import create_agent
# import warnings
# warnings.filterwarnings("ignore")

def print_lm(text: str):
        display(Markdown(text))

llm = ChatOllama(model="qwen2.5:7b", temperature=0)

@tool
def file_search(query: str) -> str:
    """Responde preguntas sobre el contenido del documento."""
    docs = retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

agent = create_agent(
    model=llm,
    tools=[file_search],
)

response = agent.invoke({
    "messages": [{"role": "user", "content": "Cual es el TTI on slow 3G que reporta el documento?"}]
})
print_lm(response["messages"][-1].content)
print('_'*50)
response = llm.invoke("Cual es el TTI on slow 3G que reporta el documento?")
print_lm(response.text)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Según el documento, el TTI (Time to Interactive) en redes 3G lentas es menor a 5 segundos.

__________________________________________________


Para responder a tu pregunta sobre el TTI (Time To Idle) en un entorno de 3G con condiciones "slow", necesitaría ver el documento específico al que te refieres. Sin tener acceso directo a ese documento, no puedo proporcionar información precisa.

Sin embargo, generalmente, el TTI on slow 3G se refiere al tiempo que un dispositivo pasa en estado de espera (idle) durante las condiciones de red lenta o con baja velocidad de datos. Este valor puede variar dependiendo del proveedor de servicios y la configuración específica de la red.

Si tienes acceso al documento, te sugiero buscar directamente la información relevante allí. Si necesitas ayuda para interpretar el contenido del documento, estaré encantado de asistirte con eso.

In [10]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "Hola cómo estas?"}]
})

print_lm(response["messages"][-1].content)

¡Hola! Estoy bien, gracias por preguntar. ¿Cómo estás tú? Si necesitas ayuda con algo, estaré encantado de asistirte.